In [2]:
!pip install --quiet pyodbc azure-identity
!pip install --quiet SQLAlchemy

In [6]:
!az login

^C


In [3]:
from azure import identity
from sqlalchemy import create_engine, MetaData, Table, select
from itertools import chain, repeat

import pandas as pd
import sqlalchemy as sa
import struct
import pyodbc
import urllib

In [4]:
sql_endpoint = "m4v3wfgtjthufjoduy5k6usfvq-6lmoii3zvfuelmwitg3eytd46e.datawarehouse.fabric.microsoft.com"
database = "MigrationLHDB"

connection_string = f"Driver={{ODBC Driver 18 for SQL Server}};Server={sql_endpoint},1433;Database={database};Encrypt=Yes;TrustServerCertificate=No"
params = urllib.parse.quote(connection_string)

In [7]:
resource_url = "https://database.windows.net/.default"
azure_credentials = identity.DefaultAzureCredential()
token_object = azure_credentials.get_token(resource_url)

# Retrieve an access token
token_as_bytes = bytes(token_object.token, "UTF-8") # Convert the token to a UTF-8 byte string
encoded_bytes = bytes(chain.from_iterable(zip(token_as_bytes, repeat(0)))) # Encode the bytes to a Windows byte string
token_bytes = struct.pack("<i", len(encoded_bytes)) + encoded_bytes # Package the token into a bytes object
attrs_before = {1256: token_bytes}  # Attribute pointing to SQL_COPT_SS_ACCESS_TOKEN to pass access token to the driver

In [9]:
print(token_object)
print(token_as_bytes)
print(encoded_bytes)
print(token_bytes)
print(attrs_before)

AccessToken(token='eyJ0eXAiOiJKV1QiLCJhbGciOiJSUzI1NiIsIng1dCI6InNNMV95QXhWOEdWNHlOLUI2ajJ4em1pazVBbyIsImtpZCI6InNNMV95QXhWOEdWNHlOLUI2ajJ4em1pazVBbyJ9.eyJhdWQiOiJodHRwczovL2RhdGFiYXNlLndpbmRvd3MubmV0IiwiaXNzIjoiaHR0cHM6Ly9zdHMud2luZG93cy5uZXQvMTRiYjJiNjctNGNkMy00MmNmLWE1YzMtYTYzYWFmNTI0NWFjLyIsImlhdCI6MTc3MTgyOTczMywibmJmIjoxNzcxODI5NzMzLCJleHAiOjE3NzE4MzQ1OTQsImFjciI6IjEiLCJhaW8iOiJBWFFBaS84YkFBQUE1bm1xUnkyNVR6QUVyeUtnQkMxWnNVZW5iV0RmcklvN3VWQnhIT0dDNHJ1UGFKMzNJYVl3QXpXY0tpKzUydDdJam5FdmVwamQ4R0lZTkNXMmZhMlVnd0tMck9Qd1RNc3ZFTU1aSE5Fd2luMHJHWENybEsyUzBZb1hyWjUrQjdPckdMeERIK21HOTF4Slc4bm1kbmJnNnc9PSIsImFtciI6WyJwd2QiLCJtZmEiXSwiYXBwaWQiOiIwNGIwNzc5NS04ZGRiLTQ2MWEtYmJlZS0wMmY5ZTFiZjdiNDYiLCJhcHBpZGFjciI6IjAiLCJncm91cHMiOlsiOThjZmQxNmYtMzRmMS00ZmI2LWI3YjUtZDdhMDQwNjdlNWNmIiwiNjg5Y2VkNmQtZTJmYy00ZTQzLWFkNmQtMGQzM2JjODc2ZjkyIiwiZDljMDYzNDItOGI4OS00ZGZlLThmZjgtNmExNjEzYWVmMTUxIiwiZTE4OTlkM2EtNTNjNy00YTgzLWJiY2ItNDRhOWY2YWJjNGY2IiwiNTEyOTAzZTYtNmI2NS00OWM1LWJhZjYtY2NiZWExOGJjOWU5IiwiMDcwYWIx

In [ ]:
# build the connection
engine = sa.create_engine("mssql+pyodbc:///?odbc_connect={0}".format(params), connect_args={'attrs_before': attrs_before})

# SQL query execution using Pandas
df = pd.read_sql("select top 10 * from MigrationLHDB.dbo.FACT_CORSEC_SOCIALMEDIA_API", engine)
print(df)

ArgumentError: Could not parse SQLAlchemy URL from given URL string